In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Reinforcement learning on a language model

## Where this session fits

> **The question we start from: Part B gave policy-gradient methods for any policy. What changes when the policy is a pretrained language model?**

- **Five steps, in order**
    - step 1: the language model as a policy (state, action, logits, sampling)
    - step 2: the reward: why a reward and not a label, and where it comes from
    - step 3: a trusted starting point: supervised fine-tuning, then the reference model and the KL reward
    - step 4: one PPO step on a language model: sample, reward, credit, update
    - step 5: what this costs, which sets up the next session
- **Position: Part C (language models), session 1 of 2**
    - Part B built the policy-gradient methods up to PPO
    - C2 is GRPO, the same loop without the critic
    - the Advanced course goes deeper: reward models, RLHF (reinforcement learning from human feedback), reasoning models
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models


**By the end you can**

1. say what the state, action, episode, reward and policy are for a language model, and why sampling makes it stochastic
2. name the four things that are new compared with the earlier environments
3. name who can produce a reward: a program (a verifier), a learned reward model, a language-model judge
4. say why supervised fine-tuning comes first, and what the KL reward does
5. read one PPO step on a language model, with its per-token credit and the models it touches


# The thesis: Part B applies, and four things are new

**Why this section?** Before the details, one claim that organizes the session.

> **A language model is a policy over tokens. RL fine-tuning is the Part B policy gradient applied to it: the theorem is unchanged.**

| What is new on a language model | Consequence | Where |
|:--|:--|:--|
| 1. the policy is a pretrained network over a huge vocabulary, and the state is text | we start from a model that already works, and must keep it there | steps 1, 3 |
| 2. one episode reward for the whole completion, and restarting from a prompt costs only another sample | credit is coarse, and a **group** of $G$ completions from one prompt is cheap ($G$ is the group size) | steps 1, 4; then C2 |
| 3. generating text is already the main cost, and a critic is a second large network | the critic is the costly addition: can the baseline come without it? | step 5; then C2 |
| 4. the reward is often a proxy for what we want | the policy must stay near a model we trust: a per-token penalty for drifting from a frozen reference model (the KL reward) | steps 2, 3 |

- **The four things are the claims. The five steps are the order we take them in.**
    - the last column says which steps deal with each claim


## Step 1 of 5: a language model is a policy

**Why this section?** Part B needs a state, an action, a policy, an episode and a reward. First say what they are for a language model.

> **The state is the prompt plus the completion so far, the action is the next token, the policy is the model's next-token distribution, and a completion is one episode.**

- **state:** the prompt $\x$ and the tokens $\y_{<\tt}$ generated so far
- **action:** the next token $\y_\tt$, chosen from the vocabulary
- **policy:** $\pi_\theta(\y_\tt \mid \x, \y_{<\tt})$, the network's distribution over the next token
- **episode:** one completion: the text the model generates after the prompt, from the first token to the end
    - $\y$ denotes a completion, and $\y_\tt$ its $\tt$-th token
- **episode reward:** one number for the whole completion, arriving after the last token (step 2 says where it comes from)

**How the network's output becomes a policy: logits, temperature, sampling**

> **The network outputs one score (a logit) for each token in the vocabulary. A softmax turns the scores into the policy, and sampling from it makes the completion random.**

- the policy is $\pi_\theta(\text{token}) = \operatorname{softmax}(z / T)$, where $z$ holds the logits and $T$ is the **temperature**
    - low $T$ sharpens the distribution toward the top token; high $T$ flattens it
    - the distribution we sample from is the policy we train, so $T$ is part of the policy
- the same prompt gives a different completion each time we sample
    - that is why the policy is stochastic, and why a policy gradient applies
- restarting from a prompt needs no environment reset
    - it costs only another sample
    - so a group of $G$ completions for one prompt is cheap


## Step 2 of 5: why a reward, and not a label

**Why this section?** A policy gradient needs a number for each completion. Supervised fine-tuning (SFT) already trains a model on example completions, so why not stop there?

> **SFT sees only the example completion (the label). It cannot rate any other answer. A reward rates every answer, including ones better than the label.**

- prompt: "How are you?"; the SFT target (the label) is "I am fine, thank you"

| Answer | In the SFT data? | Reward (made up, for illustration) |
|:--|:--|:--|
| The sky is blue | no, SFT is silent | low |
| I am fine | no, SFT is silent | medium |
| I am fine, thank you | **yes: the label** | high |
| I am fine, thank you, and how are you? | no, SFT is silent | **higher still** |

- SFT can only pull the model toward the label
    - it has no way to say that "I am fine" is acceptable, or that the last answer is richer
- a reward ranks answers the label does not contain: someone, or something, must produce those ratings
- the next slide asks who


## Step 2 of 5: where the reward comes from

**Why this section?** A reward is a number for each completion. Someone or something must produce it.

> **A reward can come from a program (a verifier), a learned reward model, or a language-model judge.**

| Reward produced by | Example | Strength | Weakness |
|:--|:--|:--|:--|
| a **program** (a **verifier**): RLVR, reinforcement learning with verifiable rewards | the final number of a math problem; unit tests for code | exact on what it checks, cheap, no human rating per completion (but someone writes the checker, and it can accept wrong answers) | only for checkable tasks |
| a **learned model** (a **reward model**) | trained on human comparisons of chat answers (next slide) | covers open-ended tasks | an imperfect proxy |
| a **judge**: a language model prompted to rate or rank completions against a rubric | "which answer is more helpful?" | flexible, no training | has its own biases (for example, favoring long answers) |


**A verifier by example**

> **Using a verifier as the reward is called RLVR: reinforcement learning with verifiable rewards.**

- prompt: "What is $17 \times 24$?"
- completion A ends "so the answer is 408": the verifier gives reward $1$
- completion B ends "so the answer is 418": reward $0$
- the episode reward arrives once, after the last token
    - the verifier knows nothing about the steps before it
- a verifier checks only what its author wrote down
    - a completion that states 408 with no working at all also gets reward $1$


**A reward model learns from comparisons (a gloss)**

> **When no program can grade a completion, a reward model learns a reward from comparisons. Nothing else in this series needs preference data.**

- people can often say which of two completions is better, even when they cannot grade one
- **preference data** records comparisons: one example is a triple $(\x, \y^+, \y^-)$
    - a prompt, a preferred completion, a non-preferred one
- the **Bradley-Terry model** turns two rewards into a probability that one completion is preferred
    - $P(\y^+ \succ \y^-) = \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
    - $\succ$ reads "is preferred to"
    - the **sigmoid** $\sigma$ turns the reward difference into a probability between $0$ and $1$
    - equal rewards give $\frac{1}{2}$, and a larger gap in favor of $\y^+$ pushes it toward $1$
- it is a **classifier**: given the pair, predict which completion won
    - fitting $\psi$ is ordinary classification on many such pairs
- fitting the **reward model** $\rew_\psi$ (parameters $\psi$, not the critic's $\phi$) to many comparisons gives one reward per completion
- that reward is the same kind a verifier gives
- the data, the training and the full pipeline, **RLHF** (reinforcement learning from human feedback), are in the Advanced course


**The reward is only a proxy**

- the reward is only a proxy for what we want, so the policy can learn to exploit it
    - **reward hacking** is getting a high reward without the quality the reward was meant to measure
    - Microduck in the A0 tour found an *unexpected strategy*
    - that counts as hacking only if the reward is earned in a way the designers did not want
    - for example, a model trained against a reward model may learn long, confident answers that earn high rewards but are not better
- a verifier is a proxy too: it checks only what its author wrote down


## Step 3 of 5: start from a model we trust

**Why this section?** The reward is a proxy (previous slide), so where RL starts matters. A first condition: the reward must be reachable. The same model then serves as the reference for the KL term, next.

> **RL can only amplify what the model already does. Supervised fine-tuning comes first to raise the probability of the wanted completions from almost zero to "sometimes".**

- **supervised fine-tuning** (SFT) trains on example completions: every token has a target
    - RL has a reward instead of a target
- RL **raises the probability of completions the model already produces** some of the time and that earn reward
- if no sampled completion earns reward (a binary verifier reward), there is no signal
    - in GRPO (C2), a group with identical rewards (all failures, or all successes) gives advantage $0$
- so a model that rarely produces the required format is unlikely to find it by RL
    - **SFT raises that probability first, so RL has something to amplify**
- this is a condition, not a law
    - a strong base model may already succeed often enough to start RL directly (some recent reasoning models did)
- SFT also supplies the fluent reference model for the KL reward


**Before spending compute**

- a check: sample a group of $G$ completions for a few prompts ($G$ is the group size, not the return of Part B)
- many groups should contain at least one success (a *hit*) and at least one failure
    - with success chance $p$ per completion, a group contains a hit with probability $1 - (1-p)^G$
        - about $G p$ when $G p$ is small
        - for $p = 0.01$, $G = 8$: about $8\%$
    - so most groups have no success at all, and RL has almost no signal: start from a better model


## Step 3 of 5: the reference model and the KL reward

**Why this section?** SFT gave a good starting model, and the reward is a proxy that can pull the policy away from it. We want to improve without wandering.

> **In RLHF (reinforcement learning from human feedback: RL against a learned reward model), PPO puts the KL penalty in the reward: every token pays $\beta$ times the log-ratio of the policy to a frozen reference model.**

- the **reference model** $\pi_{\text{ref}}$ is a frozen copy of the model before RL (for example the supervised fine-tuned model)
- the reward at token $\tt$ will have two parts
    - **the KL term, at every token:** $-\beta \log \dfrac{\pi_\theta(\y_\tt \mid \x, \y_{<\tt})}{\pi_{\text{ref}}(\y_\tt \mid \x, \y_{<\tt})}$
        - zero when the policy and the reference agree on the token
        - negative when the policy likes the token more than the reference does, positive when it likes it less
        - **KL divergence** (B4) measures how different two distributions are
        - summed over a sampled completion, the total penalty estimates $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$ for that completion
    - **the episode reward $\rew(\x, \y)$, at the last token only**


**The KL is a dense, per-step reward**

<img src="images/c1_token_rewards.png" width=85%>

- the episode reward arrives once, at the end; the KL term arrives at every token
- so the critic and GAE of Part B take it as an ordinary reward at each step
    - it is computed from the sampling policy and treated as a constant
- large $\beta$: stay close to the reference; small $\beta$: chase the reward
- it limits how far the policy can drift toward a hack, and tends to keep the text near the reference's fluency
- it does not prevent hacks that stay close to the reference
- the per-token reward is the KL term, plus the episode reward at the last token
    - step 2's reward is this episode reward alone

**Where the KL goes: the same anchor in the reward (here) or in the loss (C2)**

> **A KL term to a frozen reference model can sit in the reward (PPO, this session) or in the loss (GRPO, C2). Either way it anchors the whole run.**

| Placement | Session | KL between | Role |
|:--|:--|:--|:--|
| in the **reward** | C1 (PPO) | current policy and frozen reference | anchors the whole run |
| in the **loss** | C2 (GRPO) | current policy and frozen reference | the same anchor, different place |
| in the **objective** | B4 (an option) | current policy and old policy (or a reference) | limits one update |

- the reference never moves, so the anchor holds across the whole run
- clipping (B4) also limits one update, from the old policy to the new one: a different job
- the reference KL is not in the original PPO paper; it comes from RLHF practice


## Step 4 of 5: one PPO step on a language model

**Why this section?** Steps 1 to 3 supplied the pieces: a policy, a reward, and a reference model. This is the PPO loop of Part B (B4) with the language-model names filled in.

> **One PPO step: sample completions, reward them (episode reward plus the KL term), turn that into per-token advantages with a critic, and take a clipped step.**

- **Sample:** for a batch of prompts, draw completions $\y$ from the current policy $\pi_\theta$ (it becomes the old policy)
- **Reward:** the episode reward from the verifier, reward model or judge, plus the KL term at every token (step 3)
- **Credit:** a critic $\statevalfun_\phi$ values every prefix, and GAE (B3) turns that into an advantage for every token (next slide)
- **Update:** a clipped policy-gradient step (B4), repeated for $K$ epochs on the same batch
    - it raises the probability of tokens with a positive advantage and lowers it for the rest
    - the critic is trained on the same batch, by regression toward the TD target of B3
- repeat with fresh samples


**Credit: the critic turns one episode reward into a weight for every token**

> **The policy gradient needs a weight for each token, but the episode reward arrives once. The critic's TD error $\delta_\tt$ (B3) gives each token its own credit.**

<img src="images/c1_credit.png" width=62%>

- B3 drew this idea on a language model (the actor-critic figure)
    - here it has numbers
- the critic's value of each prefix estimates the episode reward from there
- the **surprise** $\delta_\tt$ is the reward, plus the value of the next prefix, minus the value of this one
    - the example: rewards $0, 0, 0, 0.2$; critic values of the prefixes $0.5, 0.9, 0.9, 0.9$
    - the critic's $0.9$ is too high for a completion that earns $0.2$: an inaccurate critic (B3)
    - discount $\disc = 1$, to keep the arithmetic plain
    - the value after the last token is $0$
    - the example leaves out the KL term

**Two settings of the GAE dial (B3) on the same tokens**

- the dial is the one drawn in B3 ("the estimator dial")
    - here it runs on the numbers of the previous slide

> **GAE's $\lambda$ sets how much of the later surprise a token takes. $\lambda = 0$ credits its own step; $\lambda = 1$ charges it with the whole shortfall.**

- **$\lambda = 0$:** the advantage is the token's own surprise: $+0.4,\ 0,\ 0,\ -0.7$
    - token 1 is credited for a good move, though the completion ended below expectation
- **$\lambda = 1$:** the advantage adds all later surprises: $-0.3,\ -0.7,\ -0.7,\ -0.7$
    - it equals the episode reward minus the critic's value of the prefix
    - tokens 2 to 4 are blamed equally, and the critic acts only as a baseline
- in practice $\lambda$ lies between the two
- without a critic, all four tokens would get the same weight (B2)


**The models a PPO step touches**

> **A PPO step on a language model touches up to four networks: the actor and the critic are trained; the reference and the reward model are frozen.**

<img src="images/c1_four_models.png" width=90%>

- the critic values every prefix of a long completion, and is often about as large as the actor
- the reference is a frozen copy of the starting model (step 3)
- the reward model is absent when the reward comes from a verifier
- the old actor is not a model: only the log-probabilities saved while sampling
- GRPO (C2) removes only the critic


## Step 5 of 5: what this costs, and what can go wrong

**Why this section?** The costs that motivate the next session and the Advanced course.

> **The cost the next session removes is the critic: a second large network that values every prefix of every completion. The reference and the reward model stay.**

- **the critic is the added cost:** a second large network to train, store and run, and its mistakes become bias (B3)
- **generation is already the main cost:** many long completions per prompt, every iteration
- **the reward is a proxy** (step 2): it can be hacked, and the KL term only limits the drift
- **sparse credit:** one reward for a long completion


## Aside: the probability of a whole completion

- the policy factorizes over tokens: $\pi_\theta(\y \mid \x) = \prod_{\tt=1}^{|\y|} \pi_\theta(\y_\tt \mid \x, \y_{<\tt})$
- here $\y$ is a whole completion (step 1)
- this is the probability of the *entire* episode
- the policy-gradient methods of Part B work token by token and do not need it
- methods that compare whole completions (in the Advanced course) do


# What we established, and what it sets up

- **Established**
    - Part B applies to a language model, and four things are new
        - a pretrained token policy, one episode reward per completion, costly generation, a proxy reward
    - **step 1:** a language model is a policy
        - state = prompt plus completion so far, action = next token
        - logits and temperature give the distribution we sample
    - **step 2:** a reward rates what a label cannot
        - it comes from a program, a learned reward model or a judge, each a proxy that can be hacked
    - **step 3:** supervised fine-tuning comes first
        - the KL penalty is a per-token reward to a frozen reference model, which keeps the policy from wandering
    - **step 4:** one PPO step: sample, reward, per-token advantages from a critic (the $\lambda$ dial), a clipped step
    - **step 5:** the cost is the critic, a second large network
- **Sets up**
    - **the limitation:** the critic is a second large network
    - next session (C2): GRPO, a baseline from a group of completions, with no critic
    - the Advanced course: reward models and RLHF, PPO and GRPO in depth, reasoning models
